## ToyAIKit

The handwritten agent loop from the previous lesson is educational but repetitive. Every time you build a new agent, you'd write the same while-loop, the same function-call handling, the same message management.

ToyAIKit wraps this pattern so you can focus on tools, prompts, and behavior. We built it together in a DataTalks.Club workshop a while back. It does the same thing as our handwritten loop with less boilerplate. If you open its runners code, you'll find the same while True loop we wrote by hand.

I use it here on purpose, because I don't want to pick a winner among the production frameworks. ToyAIKit is small and easy to read, so when something breaks you can see exactly what happened. That makes it handy for developing and debugging locally before you go to production.

One caveat. ToyAIKit is a teaching and experimentation library, and it is NOT meant for production use. We use it because it's minimal and you can see what it does.

Install it:

In [1]:
!uv add toyaikit

Resolved 129 packages in 1ms
Checked 125 packages in 1ms


Import the classes we need:

In [2]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

Repeat the definitions of search() and search_tool for local use...

In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [6]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

### Letting ToyAIKit generate the schema

Writing that schema by hand is annoying, and we don't want to do it for every function. So we don't have to.

If we add a type hint and a docstring to search, ToyAIKit reads them and derives the schema for us:

In [7]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

Now, register it without passing a schema:

In [8]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [9]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]